<a href="https://colab.research.google.com/github/Debojyoti-Shaw/-Ultron_1.0/blob/main/SIH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import pandas as pd
import requests

# ==========================================
# 1. API CONFIGURATION
# ==========================================
API_KEY = "579b464db66ec23bdd000001112dc93fc56f4921657b7621c8a37a3e"
# Resource ID for Agmarknet Daily Wholesale Commodity Prices
RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

# Request parameters
params = {
    "api-key": API_KEY,
    "format": "json",
    "offset": 0,
    "limit": 50,  # Number of records to fetch
    "filters[state]": "West Bengal",  # Change or remove to fetch all India data
    "filters[commodity]": "Potato",  # Filter by crop
}
print("Fetching real-time data from Agmarknet (data.gov.in)...")

try:
    response = requests.get(BASE_URL, params=params, timeout=10)

    if response.status_code == 200:
        raw_data = response.json()
        records = raw_data.get("records", [])

        if records:
            df = pd.DataFrame(records)
            columns_mapping = {
                "state": "State",
                "district": "District",
                "market": "Market",
                "commodity": "Commodity",
                "variety": "Variety",
                "arrival_date": "Date",
                "min_price": "MinPrice",
                "max_price": "MaxPrice",
                "modal_price": "ModalPrice",
            }
            available_cols = [c for c in columns_mapping.keys() if c in df.columns]
            df = df[available_cols].rename(columns=columns_mapping)
            price_cols = ["MinPrice", "MaxPrice", "ModalPrice"]
            for col in price_cols:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

            df.to_csv("agmarknet_live_data.csv", index=False)

            print(f"\nSuccessfully fetched {len(df)} live records!")
            print("\nPreview of fetched data:")
            print(df.head(10))
        else:
            print("Request succeeded, but no records were returned. Try broadening your filters.")
    else:
        print(f"Failed to fetch data. HTTP Status Code: {response.status_code}")
        print("Response:", response.text)

except Exception as e:
    print(f"An error occurred while connecting to the API: {e}")

Fetching real-time data from Agmarknet (data.gov.in)...
An error occurred while connecting to the API: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=10)


In [ ]:
import os
import time
import pandas as pd
import requests

# Configuration
API_KEY = "579b464db66ec23bdd000001112dc93fc56f4921657b7621c8a37a3e"
RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"
LOCAL_CACHE_FILE = "agmarknet_live_data.csv"

params = {
    "api-key": API_KEY,
    "format": "json",
    "offset": 0,
    "limit": 50,
    "filters[state]": "West Bengal",
    "filters[commodity]": "Potato",
}


def fetch_agmarknet_data(max_retries=3, timeout_seconds=30):
    """Fetches live data from data.gov.in with retries and fallback handling."""

    print("Fetching data from Agmarknet (data.gov.in)...")

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries}...")
            # Increased timeout to 30s: (5s connection timeout, 30s read timeout)
            response = requests.get(
                BASE_URL, params=params, timeout=(5, timeout_seconds)
            )

            if response.status_code == 200:
                raw_data = response.json()
                records = raw_data.get("records", [])

                if records:
                    df = pd.DataFrame(records)

                    # Standardize columns
                    columns_mapping = {
                        "state": "State",
                        "district": "District",
                        "market": "Market",
                        "commodity": "Commodity",
                        "variety": "Variety",
                        "arrival_date": "Date",
                        "min_price": "MinPrice",
                        "max_price": "MaxPrice",
                        "modal_price": "ModalPrice",
                    }

                    available_cols = [
                        c for c in columns_mapping.keys() if c in df.columns
                    ]
                    df = df[available_cols].rename(columns=columns_mapping)

                    # Save fresh copy locally for cache/fallback
                    df.to_csv(LOCAL_CACHE_FILE, index=False)
                    print(
                        f"\nSuccess! Fetched and cached {len(df)} live records."
                    )
                    return df

        except (requests.exceptions.RequestException, Exception) as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(2)  # Wait 2 seconds before retrying

    # --- FALLBACK MECHANISM FOR SIH DEMO ---
    print("\n Live API unavailable or timed out.")
    if os.path.exists(LOCAL_CACHE_FILE):
        print("Using locally cached dataset for seamless execution...")
        return pd.read_csv(LOCAL_CACHE_FILE)
    else:
        print(
            "No local cache found. Please run the synthetic generator script to create offline fallback data."
        )
        return None


# Execute
df_live = fetch_agmarknet_data()

if df_live is not None:
    print("\nData Preview:")
    print(df_live.head(10))

Fetching data from Agmarknet (data.gov.in)...
Attempt 1/3...
Attempt 1 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)
Attempt 2/3...
Attempt 2 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)
Attempt 3/3...
Attempt 3 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)

 Live API unavailable or timed out.
No local cache found. Please run the synthetic generator script to create offline fallback data.


In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd

# Generate 15 days of historical Agmarknet records for West Bengal mandis
dates = [
    (datetime.now() - timedelta(days=i)).strftime("%d/%m/%Y") for i in range(15)
]
mandis = [
    ("West Bengal", "Burdwan", "Burdwan Mandi", "Potato", "Jyoti"),
    ("West Bengal", "Kolkata", "Kolkata (Koley Market)", "Potato", "Jyoti"),
    ("West Bengal", "Paschim Medinipur", "Durgapur", "Potato", "Jyoti"),
    ("West Bengal", "Hooghly", "Singur", "Potato", "Jyoti"),
    ("West Bengal", "Nadia", "Ranaghat", "Potato", "Jyoti"),
]

records = []
for date_str in dates:
    for state, district, market, commodity, variety in mandis:
        modal = random.randint(2350, 2680)
        records.append(
            {
                "State": state,
                "District": district,
                "Market": market,
                "Commodity": commodity,
                "Variety": variety,
                "Date": date_str,
                "MinPrice": modal - random.randint(40, 80),
                "MaxPrice": modal + random.randint(40, 80),
                "ModalPrice": modal,
            }
        )

# Save as the exact local cache file name expected by the loader
df_cache = pd.DataFrame(records)
df_cache.to_csv("agmarknet_live_data.csv", index=False)

print(
    f"Cache file 'agmarknet_live_data.csv' created with {len(df_cache)} records!"
)
print("\nDataset Preview:")
print(df_cache.head(10))

Cache file 'agmarknet_live_data.csv' created with 75 records!

Dataset Preview:
         State           District                  Market Commodity Variety  \
0  West Bengal            Burdwan           Burdwan Mandi    Potato   Jyoti   
1  West Bengal            Kolkata  Kolkata (Koley Market)    Potato   Jyoti   
2  West Bengal  Paschim Medinipur                Durgapur    Potato   Jyoti   
3  West Bengal            Hooghly                  Singur    Potato   Jyoti   
4  West Bengal              Nadia                Ranaghat    Potato   Jyoti   
5  West Bengal            Burdwan           Burdwan Mandi    Potato   Jyoti   
6  West Bengal            Kolkata  Kolkata (Koley Market)    Potato   Jyoti   
7  West Bengal  Paschim Medinipur                Durgapur    Potato   Jyoti   
8  West Bengal            Hooghly                  Singur    Potato   Jyoti   
9  West Bengal              Nadia                Ranaghat    Potato   Jyoti   

         Date  MinPrice  MaxPrice  ModalPrice  
0 

In [ ]:
import os
import time
import pandas as pd
import requests

# Configuration
API_KEY = "579b464db66ec23bdd000001112dc93fc56f4921657b7621c8a37a3e"
RESOURCE_ID = "9ef84268-d588-465a-a308-a864a43d0070"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"
LOCAL_CACHE_FILE = "/content/agmarknet_live_data.csv"

params = {
    "api-key": API_KEY,
    "format": "json",
    "offset": 0,
    "limit": 50,
    "filters[state]": "West Bengal",
    "filters[commodity]": "Potato",
}


def fetch_agmarknet_data(max_retries=3, timeout_seconds=30):
    """Fetches live data from data.gov.in with retries and fallback handling."""

    print("Fetching data from Agmarknet (data.gov.in)...")

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries}...")
            # Increased timeout: (5s connection timeout, 30s read timeout)
            response = requests.get(
                BASE_URL, params=params, timeout=(5, timeout_seconds)
            )

            if response.status_code == 200:
                raw_data = response.json()
                records = raw_data.get("records", [])

                if records:
                    df = pd.DataFrame(records)

                    # Standardize columns
                    columns_mapping = {
                        "state": "State",
                        "district": "District",
                        "market": "Market",
                        "commodity": "Commodity",
                        "variety": "Variety",
                        "arrival_date": "Date",
                        "min_price": "MinPrice",
                        "max_price": "MaxPrice",
                        "modal_price": "ModalPrice",
                    }

                    available_cols = [
                        c for c in columns_mapping.keys() if c in df.columns
                    ]
                    df = df[available_cols].rename(columns=columns_mapping)

                    # Save fresh copy locally for cache/fallback
                    df.to_csv(LOCAL_CACHE_FILE, index=False)
                    print(
                        f"\nSuccess! Fetched and cached {len(df)} live records."
                    )
                    return df

        except (requests.exceptions.RequestException, Exception) as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(2)  # Wait 2 seconds before retrying

    # --- FALLBACK MECHANISM FOR DEMO ---
    print("\n⚠️ Live API unavailable or timed out.")
    if os.path.exists(LOCAL_CACHE_FILE):
        print("Using locally cached dataset for seamless execution...")
        return pd.read_csv(LOCAL_CACHE_FILE)
    else:
        print(
            "No local cache found. Please run the synthetic generator script to create offline fallback data."
        )
        return None


# Execute
df_live = fetch_agmarknet_data()

if df_live is not None:
    print("\nData Preview:")
    print(df_live.head(10))

Fetching data from Agmarknet (data.gov.in)...
Attempt 1/3...
Attempt 1 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)
Attempt 2/3...
Attempt 2 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)
Attempt 3/3...
Attempt 3 failed: HTTPSConnectionPool(host='api.data.gov.in', port=443): Read timed out. (read timeout=30)

⚠️ Live API unavailable or timed out.
Using locally cached dataset for seamless execution...

Data Preview:
         State           District                  Market Commodity Variety  \
0  West Bengal            Burdwan           Burdwan Mandi    Potato   Jyoti   
1  West Bengal            Kolkata  Kolkata (Koley Market)    Potato   Jyoti   
2  West Bengal  Paschim Medinipur                Durgapur    Potato   Jyoti   
3  West Bengal            Hooghly                  Singur    Potato   Jyoti   
4  West Bengal              Nadia                Ranaghat    Potato   Jyoti   
5  We

In [ ]:
import math
import pandas as pd
from typing import Dict, List, Tuple

class SmartFarmerBuyerMatcher:
    def __init__(self,
                 w_price: float = 0.35,
                 w_dist: float = 0.25,
                 w_trust: float = 0.20,
                 w_qty: float = 0.20,
                 max_distance_km: float = 300.0,
                 transport_cost_per_tonne_km: float = 3.5):

        self.w_p = w_price
        self.w_d = w_dist
        self.w_t = w_trust
        self.w_q = w_qty
        self.max_d = max_distance_km
        self.transport_cost_per_tonne_km = transport_cost_per_tonne_km

    @staticmethod
    def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        """Calculates Great-Circle distance in km between two GPS coordinates."""
        R = 6371.0 # Earth radius in km
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)

        a = (math.sin(dlat / 2)**2 +
             math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2)
        c = 2 * math.asin(math.sqrt(a))
        return R * c

    def compute_match(self, lot: pd.Series, buyer: pd.Series) -> Dict[str, float]:
        """Computes sub-scores and net realization for a single Farmer Lot - Buyer pair."""
        # 1. Distance & Score
        dist_km = self.haversine_distance(lot['lat'], lot['lon'], buyer['lat'], buyer['lon'])
        s_dist = max(0.0, 100.0 * (1.0 - (dist_km / self.max_d)))

        # 2. Price Score
        expected_p = lot['expected_price']
        offered_p = buyer['offered_price']
        s_price = min(100.0, (offered_p / expected_p) * 100.0) if expected_p > 0 else 0.0

        # 3. Quantity Score
        q_lot = lot['quantity_tonnes']
        q_req = buyer['req_quantity']
        s_qty = (min(q_lot, q_req) / max(q_lot, q_req)) * 100.0 if max(q_lot, q_req) > 0 else 0.0

        # 4. Trust Score
        s_trust = float(buyer['trust_score'])

        # Composite Match Score
        total_score = (self.w_p * s_price) + (self.w_d * s_dist) + (self.w_t * s_trust) + (self.w_q * s_qty)

        # Financial Realization: Deduct estimated transport cost per quintal (1 tonne = 10 quintals)
        transport_cost_per_quintal = (dist_km * self.transport_cost_per_tonne_km) / 10.0
        net_price_per_quintal = offered_p - transport_cost_per_quintal

        return {
            "buyer_id": buyer['buyer_id'],
            "buyer_type": buyer['buyer_type'],
            "buyer_location": buyer['location'],
            "offered_price": offered_p,
            "distance_km": round(dist_km, 1),
            "match_score": round(total_score, 2),
            "s_price": round(s_price, 1),
            "s_dist": round(s_dist, 1),
            "s_trust": round(s_trust, 1),
            "s_qty": round(s_qty, 1),
            "net_price_quintal": round(net_price_per_quintal, 2)
        }

    def match_lot(self, lot_id: str, lots_df: pd.DataFrame, buyers_df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
        """Finds top matching buyers for a given lot ID filtered by crop grade match."""
        lot_rows = lots_df[lots_df['lot_id'] == lot_id]
        if lot_rows.empty:
            raise ValueError(f"Lot ID {lot_id} not found in database.")

        lot = lot_rows.iloc[0]

        # Filter buyers demanding the same crop and grade
        eligible_buyers = buyers_df[
            (buyers_df['req_crop'].str.lower() == lot['crop'].lower()) &
            (buyers_df['req_grade'].str.upper() == lot['grade'].upper())
        ]

        if eligible_buyers.empty:
            return pd.DataFrame()

        results = []
        for _, buyer in eligible_buyers.iterrows():
            match_res = self.compute_match(lot, buyer)
            results.append(match_res)

        matched_df = pd.DataFrame(results)
        matched_df = matched_df.sort_values(by="match_score", ascending=False).head(top_n)
        return matched_df

if __name__ == "__main__":
    # Sample Mock Data
    farmer_lots = pd.DataFrame([
        {
            "lot_id": "WB-POT-001",
            "farmer_name": "Ramesh Roy",
            "location": "Hooghly, WB",
            "lat": 22.8963,
            "lon": 88.2461,
            "crop": "Potato",
            "quantity_tonnes": 12.5,
            "grade": "A",
            "expected_price": 1500.0
        }
    ])

    buyer_demands = pd.DataFrame([
        {
            "buyer_id": "BUY-IND-01",
            "buyer_type": "FMCG Processor",
            "location": "Kolkata, WB",
            "lat": 22.5726,
            "lon": 88.3639,
            "req_crop": "Potato",
            "req_quantity": 15.0,
            "req_grade": "A",
            "offered_price": 1650.0,
            "trust_score": 94.0
        },
        {
            "buyer_id": "BUY-WHO-02",
            "buyer_type": "Wholesaler",
            "location": "Burdwan, WB",
            "lat": 23.2324,
            "lon": 87.8615,
            "req_crop": "Potato",
            "req_quantity": 10.0,
            "req_grade": "A",
            "offered_price": 1520.0,
            "trust_score": 88.0
        },
        {
            "buyer_id": "BUY-RET-03",
            "buyer_type": "Retail Chain",
            "location": "Siliguri, WB",
            "lat": 26.7271,
            "lon": 88.3953,
            "req_crop": "Potato",
            "req_quantity": 12.5,
            "req_grade": "A",
            "offered_price": 1750.0,
            "trust_score": 75.0
        }
    ])

    matcher = SmartFarmerBuyerMatcher()
    recommendations = matcher.match_lot("WB-POT-001", farmer_lots, buyer_demands, top_n=3)

    print("--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---")
    print(recommendations[['buyer_id', 'buyer_type', 'offered_price', 'distance_km', 'match_score', 'net_price_quintal']].to_string(index=False))

--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---
  buyer_id     buyer_type  offered_price  distance_km  match_score  net_price_quintal
BUY-IND-01 FMCG Processor         1650.0         38.0        92.30            1636.71
BUY-WHO-02     Wholesaler         1520.0         54.3        89.08            1501.01
BUY-RET-03   Retail Chain         1750.0        426.2        70.00            1600.82


In [ ]:
import math
import pandas as pd
from typing import Dict, List, Tuple

class SmartFarmerBuyerMatcher:
    def __init__(self,
                 w_price: float = 0.35,
                 w_dist: float = 0.25,
                 w_trust: float = 0.20,
                 w_qty: float = 0.20,
                 max_distance_km: float = 300.0,
                 transport_cost_per_tonne_km: float = 3.5):

        self.w_p = w_price
        self.w_d = w_dist
        self.w_t = w_trust
        self.w_q = w_qty
        self.max_d = max_distance_km
        self.transport_cost_per_tonne_km = transport_cost_per_tonne_km

    @staticmethod
    def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        """Calculates Great-Circle distance in km between two GPS coordinates."""
        R = 6371.0 # Earth radius in km
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)

        a = (math.sin(dlat / 2)**2 +
             math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2)
        c = 2 * math.asin(math.sqrt(a))
        return R * c

    def compute_match(self, lot: pd.Series, buyer: pd.Series) -> Dict[str, float]:
        """Computes sub-scores and net realization for a single Farmer Lot - Buyer pair."""
        # 1. Distance & Score
        dist_km = self.haversine_distance(lot['lat'], lot['lon'], buyer['lat'], buyer['lon'])
        s_dist = max(0.0, 100.0 * (1.0 - (dist_km / self.max_d)))

        # 2. Price Score
        expected_p = lot['expected_price']
        offered_p = buyer['offered_price']
        s_price = min(100.0, (offered_p / expected_p) * 100.0) if expected_p > 0 else 0.0

        # 3. Quantity Score
        q_lot = lot['quantity_tonnes']
        q_req = buyer['req_quantity']
        s_qty = (min(q_lot, q_req) / max(q_lot, q_req)) * 100.0 if max(q_lot, q_req) > 0 else 0.0

        # 4. Trust Score
        s_trust = float(buyer['trust_score'])

        # Composite Match Score
        total_score = (self.w_p * s_price) + (self.w_d * s_dist) + (self.w_t * s_trust) + (self.w_q * s_qty)

        # Financial Realization: Deduct estimated transport cost per quintal (1 tonne = 10 quintals)
        transport_cost_per_quintal = (dist_km * self.transport_cost_per_tonne_km) / 10.0
        net_price_per_quintal = offered_p - transport_cost_per_quintal

        return {
            "buyer_id": buyer['buyer_id'],
            "buyer_type": buyer['buyer_type'],
            "buyer_location": buyer['location'],
            "offered_price": offered_p,
            "distance_km": round(dist_km, 1),
            "trust_pct": f"{round(s_trust, 1)}%",  # Added formatted Trust Percentage
            "match_score": round(total_score, 2),
            "s_price": round(s_price, 1),
            "s_dist": round(s_dist, 1),
            "s_trust": round(s_trust, 1),
            "s_qty": round(s_qty, 1),
            "net_price_quintal": round(net_price_per_quintal, 2)
        }

    def match_lot(self, lot_id: str, lots_df: pd.DataFrame, buyers_df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
        """Finds top matching buyers for a given lot ID filtered by crop grade match."""
        lot_rows = lots_df[lots_df['lot_id'] == lot_id]
        if lot_rows.empty:
            raise ValueError(f"Lot ID {lot_id} not found in database.")

        lot = lot_rows.iloc[0]

        # Filter buyers demanding the same crop and grade
        eligible_buyers = buyers_df[
            (buyers_df['req_crop'].str.lower() == lot['crop'].lower()) &
            (buyers_df['req_grade'].str.upper() == lot['grade'].upper())
        ]

        if eligible_buyers.empty:
            return pd.DataFrame()

        results = []
        for _, buyer in eligible_buyers.iterrows():
            match_res = self.compute_match(lot, buyer)
            results.append(match_res)

        matched_df = pd.DataFrame(results)
        matched_df = matched_df.sort_values(by="match_score", ascending=False).head(top_n)
        return matched_df


# ==========================================
# Demonstration & Test Run
# ==========================================
if __name__ == "__main__":
    # Sample Mock Data
    farmer_lots = pd.DataFrame([
        {
            "lot_id": "WB-POT-001",
            "farmer_name": "Ramesh Roy",
            "location": "Hooghly, WB",
            "lat": 22.8963,
            "lon": 88.2461,
            "crop": "Potato",
            "quantity_tonnes": 12.5,
            "grade": "A",
            "expected_price": 1500.0
        }
    ])

    buyer_demands = pd.DataFrame([
        {
            "buyer_id": "BUY-IND-01",
            "buyer_type": "FMCG Processor",
            "location": "Kolkata, WB",
            "lat": 22.5726,
            "lon": 88.3639,
            "req_crop": "Potato",
            "req_quantity": 15.0,
            "req_grade": "A",
            "offered_price": 1650.0,
            "trust_score": 94.0
        },
        {
            "buyer_id": "BUY-WHO-02",
            "buyer_type": "Wholesaler",
            "location": "Burdwan, WB",
            "lat": 23.2324,
            "lon": 87.8615,
            "req_crop": "Potato",
            "req_quantity": 10.0,
            "req_grade": "A",
            "offered_price": 1520.0,
            "trust_score": 88.0
        },
        {
            "buyer_id": "BUY-RET-03",
            "buyer_type": "Retail Chain",
            "location": "Siliguri, WB",
            "lat": 26.7271,
            "lon": 88.3953,
            "req_crop": "Potato",
            "req_quantity": 12.5,
            "req_grade": "A",
            "offered_price": 1750.0,
            "trust_score": 75.0
        }
    ])

    matcher = SmartFarmerBuyerMatcher()
    recommendations = matcher.match_lot("WB-POT-001", farmer_lots, buyer_demands, top_n=3)

    print("--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---")
    print(recommendations[['buyer_id', 'buyer_type', 'offered_price', 'distance_km', 'trust_pct', 'match_score', 'net_price_quintal']].to_string(index=False))

--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---
  buyer_id     buyer_type  offered_price  distance_km trust_pct  match_score  net_price_quintal
BUY-IND-01 FMCG Processor         1650.0         38.0     94.0%        92.30            1636.71
BUY-WHO-02     Wholesaler         1520.0         54.3     88.0%        89.08            1501.01
BUY-RET-03   Retail Chain         1750.0        426.2     75.0%        70.00            1600.82


In [ ]:
%%bash
# 1. Install and start PostgreSQL service
apt-get update -qq
apt-get install -y postgresql postgresql-contrib -qq
service postgresql start

# 2. Configure password 'postgres' and create database 'agri_db'
sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"
sudo -u postgres psql -c "CREATE DATABASE agri_db;"

# 3. Create tables and insert mock data
sudo -u postgres psql -d agri_db -c "
CREATE TABLE IF NOT EXISTS farmer_lots (
    lot_id VARCHAR(50) PRIMARY KEY, farmer_name VARCHAR(100), location VARCHAR(100),
    lat DOUBLE PRECISION, lon DOUBLE PRECISION, crop VARCHAR(50),
    quantity_tonnes DOUBLE PRECISION, grade VARCHAR(10), expected_price DOUBLE PRECISION
);
CREATE TABLE IF NOT EXISTS buyer_demands (
    buyer_id VARCHAR(50) PRIMARY KEY, buyer_type VARCHAR(100), location VARCHAR(100),
    lat DOUBLE PRECISION, lon DOUBLE PRECISION, req_crop VARCHAR(50),
    req_quantity DOUBLE PRECISION, req_grade VARCHAR(10), offered_price DOUBLE PRECISION, trust_score DOUBLE PRECISION
);
INSERT INTO farmer_lots VALUES ('WB-POT-001', 'Ramesh Roy', 'Hooghly, WB', 22.8963, 88.2461, 'Potato', 12.5, 'A', 1500.0) ON CONFLICT DO NOTHING;
INSERT INTO buyer_demands VALUES
('BUY-IND-01', 'FMCG Processor', 'Kolkata, WB', 22.5726, 88.3639, 'Potato', 15.0, 'A', 1650.0, 94.0),
('BUY-WHO-02', 'Wholesaler', 'Burdwan, WB', 23.2324, 87.8615, 'Potato', 10.0, 'A', 1520.0, 88.0),
('BUY-RET-03', 'Retail Chain', 'Siliguri, WB', 26.7271, 88.3953, 'Potato', 12.5, 'A', 1750.0, 75.0)
ON CONFLICT DO NOTHING;
"

 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE
INSERT 0 0


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR:  database "agri_db" already exists
NOTICE:  relation "farmer_lots" already exists, skipping
NOTICE:  relation "buyer_demands" already exists, skipping


In [ ]:
import math
from typing import Dict, List, Tuple
import pandas as pd
from sqlalchemy import create_engine

# 1. PostgreSQL Connection Settings inside Colab
DB_USER = "postgres"
DB_PASSWORD = "postgres"  # Matches password configured in Cell 1
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "agri_db"

DATABASE_URL = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


def fetch_data_from_db(db_url: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
  """Fetches farmer_lots and buyer_demands directly from PostgreSQL."""
  engine = create_engine(db_url)
  farmer_lots_df = pd.read_sql("SELECT * FROM farmer_lots;", engine)
  buyer_demands_df = pd.read_sql("SELECT * FROM buyer_demands;", engine)
  print(
      " Successfully fetched records from live PostgreSQL database inside"
      " Colab!"
  )
  return farmer_lots_df, buyer_demands_df


# 2. Smart Matcher Engine
class SmartFarmerBuyerMatcher:

  def __init__(
      self,
      w_price: float = 0.35,
      w_dist: float = 0.25,
      w_trust: float = 0.20,
      w_qty: float = 0.20,
      max_distance_km: float = 300.0,
      transport_cost_per_tonne_km: float = 3.5,
  ):

    self.w_p, self.w_d, self.w_t, self.w_q = (
        w_price,
        w_dist,
        w_trust,
        w_qty,
    )
    self.max_d = max_distance_km
    self.transport_cost_per_tonne_km = transport_cost_per_tonne_km

  @staticmethod
  def haversine_distance(
      lat1: float, lon1: float, lat2: float, lon2: float
  ) -> float:
    R = 6371.0
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(
        math.radians(lat2)
    ) * math.sin(dlon / 2) ** 2
    return R * (2 * math.asin(math.sqrt(a)))

  def compute_match(
      self, lot: pd.Series, buyer: pd.Series
  ) -> Dict[str, float]:
    dist_km = self.haversine_distance(
        lot["lat"], lot["lon"], buyer["lat"], buyer["lon"]
    )
    s_dist = max(0.0, 100.0 * (1.0 - (dist_km / self.max_d)))
    s_price = (
        min(100.0, (buyer["offered_price"] / lot["expected_price"]) * 100.0)
        if lot["expected_price"] > 0
        else 0.0
    )
    s_qty = (
        min(lot["quantity_tonnes"], buyer["req_quantity"])
        / max(lot["quantity_tonnes"], buyer["req_quantity"])
    ) * 100.0
    s_trust = float(buyer["trust_score"])

    total_score = (
        (self.w_p * s_price)
        + (self.w_d * s_dist)
        + (self.w_t * s_trust)
        + (self.w_q * s_qty)
    )
    net_price_per_quintal = buyer["offered_price"] - (
        (dist_km * self.transport_cost_per_tonne_km) / 10.0
    )

    return {
        "buyer_id": buyer["buyer_id"],
        "buyer_type": buyer["buyer_type"],
        "offered_price": buyer["offered_price"],
        "distance_km": round(dist_km, 1),
        "trust_pct": f"{round(s_trust, 1)}%",
        "match_score": round(total_score, 2),
        "net_price_quintal": round(net_price_per_quintal, 2),
    }

  def match_lot(
      self,
      lot_id: str,
      lots_df: pd.DataFrame,
      buyers_df: pd.DataFrame,
      top_n: int = 3,
  ) -> pd.DataFrame:
    lot = lots_df[lots_df["lot_id"] == lot_id].iloc[0]
    eligible_buyers = buyers_df[
        (buyers_df["req_crop"].str.lower() == lot["crop"].lower())
        & (buyers_df["req_grade"].str.upper() == lot["grade"].upper())
    ]
    results = [
        self.compute_match(lot, buyer)
        for _, buyer in eligible_buyers.iterrows()
    ]
    return pd.DataFrame(results).sort_values(
        by="match_score", ascending=False
    ).head(top_n)


# 3. Fetch from DB & Run Matcher
farmer_lots, buyer_demands = fetch_data_from_db(DATABASE_URL)
matcher = SmartFarmerBuyerMatcher()
recommendations = matcher.match_lot("WB-POT-001", farmer_lots, buyer_demands)

print("\n--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---")
print(recommendations.to_string(index=False))

 Successfully fetched records from live PostgreSQL database inside Colab!

--- MATCHING RECOMMENDATIONS FOR LOT WB-POT-001 ---
  buyer_id     buyer_type  offered_price  distance_km trust_pct  match_score  net_price_quintal
BUY-IND-01 FMCG Processor         1650.0         38.0     94.0%        92.30            1636.71
BUY-WHO-02     Wholesaler         1520.0         54.3     88.0%        89.08            1501.01
BUY-RET-03   Retail Chain         1750.0        426.2     75.0%        70.00            1600.82


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

# ==========================================
# 1. SYNTHETIC AGMARKNET TIME-SERIES GENERATOR
# ==========================================
def generate_historical_mandi_data(days=365, commodity="Onion", base_price=2200):
    """
    Generates realistic daily Agmarknet market data with seasonality and noise.
    """
    np.random.seed(42)
    dates = [datetime.now().date() - timedelta(days=i) for i in range(days)]
    dates.reverse()

    # Create trend + seasonal signal + noise
    t = np.arange(days)
    seasonal = 350 * np.sin(2 * np.pi * t / 180) + 200 * np.cos(2 * np.pi * t / 365)
    trend = 1.5 * t
    noise = np.random.normal(0, 45, days)

    prices = base_price + trend + seasonal + noise

    df = pd.DataFrame({
        'date': pd.to_datetime(dates),
        'commodity': commodity,
        'modal_price': np.round(prices, 2),
        'arrival_volume_tonnes': np.random.randint(50, 500, size=days)
    })
    return df

# ==========================================
# 2. FEATURE ENGINEERING FOR XGBOOST
# ==========================================
def create_time_series_features(df):
    """
    Creates temporal, lag, and rolling window features for price forecasting.
    """
    data = df.copy()
    data = data.sort_values('date').reset_index(drop=True)

    # Time features
    data['dayofweek'] = data['date'].dt.dayofweek
    data['month'] = data['date'].dt.month
    data['dayofyear'] = data['date'].dt.dayofyear
    data['quarter'] = data['date'].dt.quarter

    # Lag features (past prices)
    for lag in [1, 2, 3, 7, 14, 30]:
        data[f'price_lag_{lag}'] = data['modal_price'].shift(lag)

    # Rolling window statistics
    data['rolling_mean_7'] = data['modal_price'].shift(1).rolling(window=7).mean()
    data['rolling_std_7'] = data['modal_price'].shift(1).rolling(window=7).std()
    data['rolling_mean_14'] = data['modal_price'].shift(1).rolling(window=14).mean()
    data['rolling_min_7'] = data['modal_price'].shift(1).rolling(window=7).min()
    data['rolling_max_7'] = data['modal_price'].shift(1).rolling(window=7).max()

    return data.dropna().reset_index(drop=True)

# ==========================================
# 3. XGBOOST FORECASTING MODEL
# ==========================================
class AgmarknetXGBForecaster:
    def __init__(self):
        self.model = xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
        self.feature_cols = [
            'dayofweek', 'month', 'dayofyear', 'quarter',
            'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7', 'price_lag_14', 'price_lag_30',
            'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_min_7', 'rolling_max_7'
        ]

    def train(self, df):
        """
        Trains XGBoost Regressor on processed historical data.
        """
        processed_df = create_time_series_features(df)
        X = processed_df[self.feature_cols]
        y = processed_df['modal_price']

        # Train model
        self.model.fit(X, y)

        # In-sample validation metrics
        y_pred = self.model.predict(X)
        mape = mean_absolute_percentage_error(y, y_pred) * 100
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        print(f"✅ XGBoost Model Trained | RMSE: ₹{rmse:.2f} | MAPE: {mape:.2f}%")

        self.last_known_data = processed_df

    def forecast_next_n_days(self, horizon_days=7):
        """
        Performs recursive 7-day multi-step price forecasting.
        """
        history = self.last_known_data.copy()
        forecasts = []
        last_date = history['date'].max()

        for day in range(1, horizon_days + 1):
            next_date = last_date + timedelta(days=day)

            # Construct row for next_date
            latest_prices = history['modal_price'].values

            feat_row = {
                'dayofweek': next_date.dayofweek,
                'month': next_date.month,
                'dayofyear': next_date.dayofyear,
                'quarter': next_date.quarter,
                'price_lag_1': latest_prices[-1],
                'price_lag_2': latest_prices[-2],
                'price_lag_3': latest_prices[-3],
                'price_lag_7': latest_prices[-7],
                'price_lag_14': latest_prices[-14],
                'price_lag_30': latest_prices[-30],
                'rolling_mean_7': np.mean(latest_prices[-7:]),
                'rolling_std_7': np.std(latest_prices[-7:]),
                'rolling_mean_14': np.mean(latest_prices[-14:]),
                'rolling_min_7': np.min(latest_prices[-7:]),
                'rolling_max_7': np.max(latest_prices[-7:]),
            }

            X_next = pd.DataFrame([feat_row])[self.feature_cols]
            predicted_price = float(self.model.predict(X_next)[0])

            forecasts.append({
                'date': next_date.strftime('%Y-%m-%d'),
                'day_offset': day,
                'predicted_price': round(predicted_price, 2)
            })

            # Append prediction to history to feed recursive lags
            new_row = pd.DataFrame([{
                'date': next_date,
                'modal_price': predicted_price
            }])
            history = pd.concat([history, new_row], ignore_index=True)

        return pd.DataFrame(forecasts)

# ==========================================
# 4. SELL-OR-STORE DECISION SUPPORT ENGINE
# ==========================================
class SellOrStoreAdvisor:
    def __init__(self, storage_cost_per_qtl_day=3.5, spoilage_risk_pct_per_day=0.002):
        """
        :param storage_cost_per_qtl_day: Daily cold storage/godown charge (in ₹ per quintal).
        :param spoilage_risk_pct_per_day: Quality degradation or weight loss factor per day.
        """
        self.storage_cost = storage_cost_per_qtl_day
        self.spoilage_rate = spoilage_risk_pct_per_day

    def evaluate_sell_window(self, current_price, forecast_df, crop_quantity_qtl):
        """
        Computes Net Realizable Profit across the 7-day forecast window.
        """
        best_day = 0
        max_net_gain = 0
        recommendations = []

        # Day 0 (Sell Today) Baseline
        recommendations.append({
            'day': 0,
            'date': 'Today',
            'raw_price': current_price,
            'storage_cost': 0.0,
            'spoilage_loss': 0.0,
            'net_realizable_price': current_price,
            'additional_gain_per_qtl': 0.0
        })

        for idx, row in forecast_df.iterrows():
            days = row['day_offset']
            pred_price = row['predicted_price']

            # Total storage cost incurred over N days
            total_storage = self.storage_cost * days

            # Estimated loss due to weight loss/degradation
            effective_price = pred_price * (1 - (self.spoilage_rate * days))

            # Net price realized after expenses
            net_price = effective_price - total_storage
            net_gain = net_price - current_price

            recommendations.append({
                'day': days,
                'date': row['date'],
                'raw_price': pred_price,
                'storage_cost': round(total_storage, 2),
                'spoilage_loss': round(pred_price - effective_price, 2),
                'net_realizable_price': round(net_price, 2),
                'additional_gain_per_qtl': round(net_gain, 2)
            })

            if net_gain > max_net_gain:
                max_net_gain = net_gain
                best_day = days

        rec_df = pd.DataFrame(recommendations)

        # Decision Threshold Logic (Require at least ₹25/qtl gain to justify holding risk)
        if max_net_gain >= 25.0:
            best_option = rec_df[rec_df['day'] == best_day].iloc[0]
            action = "HOLD & STORE"
            summary = (f"🚨 **RECOMMENDATION: HOLD FOR {best_day} DAYS**\n"
                       f"• Current Price: ₹{current_price}/qtl\n"
                       f"• Forecasted Price (Day {best_day}): ₹{best_option['raw_price']}/qtl\n"
                       f"• Net Profit Gain (after storage & shrinkage): +₹{max_net_gain:.2f}/qtl\n"
                       f"• Total Projected Net Gain for {crop_quantity_qtl} qtl: ₹{max_net_gain * crop_quantity_qtl:,.2f}")
        else:
            action = "SELL TODAY"
            summary = (f"⚡ **RECOMMENDATION: SELL IMMEDIATELY**\n"
                       f"• Current Price: ₹{current_price}/qtl\n"
                       f"• Expected price gains do not exceed storage & spoilage risks.")

        return action, summary, rec_df

# ==========================================
# 5. INTEGRATED PIPELINE EXECUTION
# ==========================================
if __name__ == "__main__":
    print("=" * 60)
    print("🌾 AGRI-LINK XGBOOST PRICE FORECASTER & DECISION ENGINE 🌾")
    print("=" * 60)

    # Step 1: Load/Generate historical Agmarknet daily price data
    print("\n1️⃣ Ingesting historical Agmarknet mandi data...")
    raw_data = generate_historical_mandi_data(days=365, commodity="Onion", base_price=2100)

    # Step 2: Train XGBoost Model
    print("\n2️⃣ Training XGBoost Time-Series Regressor...")
    forecaster = AgmarknetXGBForecaster()
    forecaster.train(raw_data)

    # Step 3: Produce 7-Day Forecast
    print("\n3️⃣ Generating 7-Day Price Forecast...")
    forecast_df = forecaster.forecast_next_n_days(horizon_days=7)
    print("\n--- 7-Day Price Forecast Table ---")
    print(forecast_df.to_string(index=False))

    # Step 4: Run Sell-or-Store Decision Engine
    print("\n4️⃣ Evaluating Sale Window (Storage vs Immediate Sale)...")
    current_mandi_price = float(raw_data['modal_price'].iloc[-1])
    farmer_quantity_quintals = 50 # 5 Tons of produce

    advisor = SellOrStoreAdvisor(storage_cost_per_qtl_day=4.0, spoilage_risk_pct_per_day=0.003)
    action, summary, decision_matrix = advisor.evaluate_sell_window(
        current_price=current_mandi_price,
        forecast_df=forecast_df,
        crop_quantity_qtl=farmer_quantity_quintals
    )

    print("\n" + summary + "\n")
    print("--- Detailed Financial Analysis (₹/Quintal) ---")
    print(decision_matrix[['day', 'date', 'raw_price', 'storage_cost', 'net_realizable_price', 'additional_gain_per_qtl']].to_string(index=False))

🌾 AGRI-LINK XGBOOST PRICE FORECASTER & DECISION ENGINE 🌾

1️⃣ Ingesting historical Agmarknet mandi data...

2️⃣ Training XGBoost Time-Series Regressor...
✅ XGBoost Model Trained | RMSE: ₹11.34 | MAPE: 0.39%

3️⃣ Generating 7-Day Price Forecast...

--- 7-Day Price Forecast Table ---
      date  day_offset  predicted_price
2026-08-29           1          2914.49
2026-08-30           2          2886.53
2026-08-31           3          2910.12
2026-09-01           4          2905.38
2026-09-02           5          2909.72
2026-09-03           6          2906.78
2026-09-04           7          2903.29

4️⃣ Evaluating Sale Window (Storage vs Immediate Sale)...

⚡ **RECOMMENDATION: SELL IMMEDIATELY**
• Current Price: ₹2925.74/qtl
• Expected price gains do not exceed storage & spoilage risks.

--- Detailed Financial Analysis (₹/Quintal) ---
 day       date  raw_price  storage_cost  net_realizable_price  additional_gain_per_qtl
   0      Today    2925.74           0.0               2925.74      